# 54 — Doc2Query catalog generation (one-time)

**Goal**: for each of the 47K tracks, generate 5 conversational queries an actual user might use to ask for it. Store as a local parquet for tomorrow's BM25 corpus enrichment.

**Why this should help** (per Phase 0 diagnostic): 78.5% of all misses are `not_in_either` — gold absent from BOTH BM25 top-100 AND dense top-100. The candidate set itself is too narrow. Doc2query directly enlarges what the index can match against by giving each track multiple natural-language descriptions.

**Workflow**:
1. Smoke run (20 tracks, ~1 min) — spot-check query quality before committing the full run
2. If quality looks reasonable, kick off full generation (47K tracks, ~2–3 hr on L4)
3. Parquet writes to `experiments/cache/doc2query/<safe_model>/queries.parquet` AND is auto-copied to Drive for safety

**Resume support**: the script will skip tracks already in the output parquet, so a Colab disconnect mid-run is recoverable — just re-run the full-generation cell.

**Wallclock estimates** (Qwen-2.5-1.5B-Instruct, batch=16, ~200 max_new_tokens, temperature 0.8):
| GPU | Smoke (20 tracks) | Full (47K tracks) |
|---|---|---|
| A100 / Blackwell | ~30 sec | ~1.5 hr |
| L4 (24 GB) | ~60 sec | ~3 hr |
| T4 (16 GB) | ~2 min | ~6 hr |

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth + Drive mount (parquet output gets backed up to Drive automatically).
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('NO HF_TOKEN — set it in Colab secrets before cell 5.', e)

from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

# Symlink the doc2query output dir to Drive so generation survives session resets.
DRIVE_DOC2QUERY = '/content/drive/MyDrive/recsys2026_doc2query_cache'
os.makedirs(DRIVE_DOC2QUERY, exist_ok=True)
REPO_DOC2QUERY = '/content/recsys2026/experiments/cache/doc2query'
os.makedirs(os.path.dirname(REPO_DOC2QUERY), exist_ok=True)
if os.path.islink(REPO_DOC2QUERY) or os.path.exists(REPO_DOC2QUERY):
    !rm -rf {REPO_DOC2QUERY}
!ln -s {DRIVE_DOC2QUERY} {REPO_DOC2QUERY}
print('symlinked', REPO_DOC2QUERY, '->', DRIVE_DOC2QUERY)

In [ ]:
# 4) Install deps.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml pyarrow

In [ ]:
# 5) SMOKE — generate for 20 tracks, ~1 min on L4. Spot-check quality before committing 3 hr.
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
!python scripts/doc2query_generate.py \
    --model {MODEL} \
    --n-queries 5 \
    --batch-size 16 \
    --max-tracks 20

# Inspect the smoke output
import pandas as pd
safe_model = MODEL.replace('/', '_')
df = pd.read_parquet(f'experiments/cache/doc2query/{safe_model}/queries.parquet')
print(f'\n=== SMOKE OUTPUT: {len(df)} tracks ===\n')
for _, row in df.head(3).iterrows():
    print(f"track_id: {row['track_id']}")
    for q in row['synthetic_queries']:
        print(f'  - {q}')
    print()

### Spot-check the smoke output above

Look at the 3 sample tracks printed by cell 5. Each should have ~5 queries that:
- Read like things a real user would say ("play me something for a rainy Sunday", not "track_name: Bohemian Rhapsody")
- Vary across mood, occasion, era, style
- Are 5–15 words each
- Don't include literal metadata field names (no "Title: X")

If quality looks good → run cell 6 below for the full 47K generation.

If quality looks bad → don't run cell 6. Ping me with the smoke output and we'll tune the prompt or model choice.

In [ ]:
# 6) FULL generation — 47K tracks, ~2-3 hr on L4. Resume-safe: skips tracks already in parquet.
# Output auto-saved to Drive every 500 tracks, so a session disconnect doesn't lose work.
# Just re-run this cell after disconnect; --resume picks up where it stopped.
!python scripts/doc2query_generate.py \
    --model {MODEL} \
    --n-queries 5 \
    --batch-size 16 \
    --resume

In [ ]:
# 7) Verify final parquet on Drive.
import pandas as pd
safe_model = MODEL.replace('/', '_')
drive_path = f'/content/drive/MyDrive/recsys2026_doc2query_cache/{safe_model}/queries.parquet'
df = pd.read_parquet(drive_path)
print(f'rows: {len(df)} (target: 47,071)')
print(f'avg queries per track: {df["synthetic_queries"].apply(len).mean():.2f}')
print(f'tracks with < 3 queries: {(df["synthetic_queries"].apply(len) < 3).sum()}')
print(f'parquet size: {os.path.getsize(drive_path) / 1024 / 1024:.1f} MB')
print(f'\nLocation on Drive: {drive_path}')
print('Ready for tomorrow: notebook 55 will use this to build the doc2query-enriched BM25 index.')